In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
from privacy_analysis.compute_privacy_sgm import compute_dp_sgd_privacy
from DP_ACE_Materials.DP_ACE import DP_ACE

In [61]:
imdb = load_dataset("imdb")
espam = load_dataset("SetFit/enron_spam")

max_imdb_contribution = len(imdb["train"])
max_espam_contribution = len(espam["train"])

total_if_imdb_maxed = int(len(imdb["train"]) / 0.96)
total_if_espam_maxed = int(len(espam["train"]) / 0.04)
total_desired_size = min(total_if_imdb_maxed, total_if_espam_maxed)
imdb_size = int(0.96 * total_desired_size)
espam_size = int(0.04 * total_desired_size)

np.random.seed(42)
imdb_indices = np.random.choice(len(imdb["train"]), size=len(imdb), replace=False)
espam_indices = np.random.choice(len(espam["train"]), size=6250, replace=False)

imdb_subset = imdb["train"]
espam_subset = espam["train"].select(espam_indices)

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
07/31/2025 18:14:52:WARNING:HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
Retrying in 1s [Retry 1/5].
07/31/2025 18:14:52:WARNING:Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
07/31/2025 18:14:53:WARNING:HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
Retrying in 2s [Retry 2/5].
07/31/2025 18:14:53:WARNING:Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
07/31/2025 18:14:55:WARNING:HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
Retrying in 4s [Retry 3/5].
07/31/2025 18:14:55:WARNING:Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting H

In [62]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

min_freq = 3
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)[:max_len]
    bow_vector = torch.zeros(len(vocab), dtype=torch.float)
    for token in tokens:
        idx = vocab.get(token, UNK_IDX)
        bow_vector[idx] += 1
    return bow_vector

In [63]:
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, text_field, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len
        self.text_field = text_field

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = text_pipeline(item[self.text_field], max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

In [64]:
train_loader_public = DataLoader(
    TextDataset(espam_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

train_loader_private = DataLoader(
    TextDataset(imdb_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

test_loader = DataLoader(
    TextDataset(imdb["test"], label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=False, collate_fn=collate_batch
)

In [65]:
class Model(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)

    def forward(self, x):
        return self.fc(x)

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
privacy_engine = PrivacyEngine()
gc.collect()

115

In [ ]:
train_losses, train_accuracies, test_losses, test_accuracies = DP_ACE(
                                                                        model=model,
                                                                        D_loader=train_loader_private,
                                                                        D_s_loader=train_loader_public,
                                                                        test_loader=test_loader,
                                                                        lr=0.01, 
                                                                        sigma=0.716, 
                                                                        l=criterion,
                                                                        T=15,
                                                                        delta=1e-5
                                                                      )

DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 391 steps satisfies differential privacy with eps = 1.95 and delta = 1e-05.
The optimal RDP order is 6.0.
Epoch [1/15] ... Training Loss: 0.7507 ... Training Accuracy: 59.10% ... Test Loss: 0.6651 ... Test Accuracy: 59.91% ... Epsilon: 1.9501 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 782 steps satisfies differential privacy with eps = 2.12 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [2/15] ... Training Loss: 0.7244 ... Training Accuracy: 61.08% ... Test Loss: 0.6509 ... Test Accuracy: 62.04% ... Epsilon: 2.1208 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 1172 steps satisfies differential privacy with eps = 2.19 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [3/15] ... Training Loss: 0.7195 ... Training Accuracy: 63.12% ... Test Loss: 0.6496 ... Test Accuracy: 62.50% ... Epsilon: 2.1915 ..